# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Predicting Search Performance Decline for Content Prioritization**

A decision-support study using search and engagement signals from the FlyRank ML Internship dataset

## Abstract

This study asks whether historical search and engagement signals can help prioritize content pages for human review when they are at risk of future search-performance decline. Using the FlyRank ML Internship dataset, I formulate the problem as a binary ranking task in which a page is labeled as declining when its future-period Google Search impressions fall by more than 20%. I compare a simple rule-based baseline with Logistic Regression using five features measured before the outcome window and evaluate the model using a client-grouped held-out split. The Logistic Regression model achieved a Precision@50 of 0.66, compared with 0.44 for the baseline under the same evaluation design. The result suggests that historical search and engagement signals can provide useful directional decision support for prioritizing human review, but the model does not establish causality or determine automatically which pages should be refreshed.

## 1. Question

*The research question and the decision it supports.*

## 1. Introduction / Problem Statement

Content teams often have more pages to review than they can manually inspect. A practical question is therefore whether historical performance signals can help prioritize which pages deserve attention first.

This study investigates whether search and engagement signals observed before an outcome period can be used to rank pages according to their risk of future search-performance decline.

The decision supported by the analysis is prioritization: which pages should a content specialist review first?

The intended action is not automatic editing. A high-ranked page is a candidate for human investigation, which may result in a refresh, monitoring, or no action.

The central research question is:

> Can historical search and engagement signals improve the prioritization of pages for human review compared with a simple rule-based baseline?

This framing deliberately treats the model as decision support rather than an automated content optimization system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

The analysis uses the publicly released FlyRank ML Internship dataset available through Hugging Face.

The relevant warehouse data comes from the content daily performance data used throughout the internship exercises.

The analysis uses a March 2026 decision window and an April 2026 outcome window.

Each modeling row represents a client-content page pair aggregated over the decision window.

The analysis uses the following historical features:

- Google Search Console impressions
- Google Search Console clicks
- Google Search Console average position
- Organic sessions
- GA4 engaged sessions

The outcome is calculated from the subsequent period rather than from the feature window.

Pages without the required data availability signal were excluded using the warehouse availability fields. Rows with missing values required by the model were also excluded.

The analysis deliberately excludes client names, page URLs, private queries, and other identifying information. Client and content identifiers are retained only as hashed identifiers for grouping and reproducibility within the dataset.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### ML task

The task is binary risk scoring used for ranking.

For each page, the model estimates the probability that the page will experience a greater-than-20% decline in Google Search impressions in the following period.

The resulting probability is used to rank pages rather than as a definitive classification of whether a page should be changed.

### Label definition

The decline label is:

`1` if future-period impressions decline by more than 20% relative to the decision period, otherwise `0`.

The future-period performance is used only to construct the label and is not available as a feature at the decision moment.

### Model

I use Logistic Regression because the target is binary and the model produces interpretable probabilities that can be used to rank pages.

The five input features are:

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `sessions_organic`
5. `ga4_engaged_sessions`

The features are standardized before fitting the Logistic Regression model.

### Baseline

The Week-4 baseline is a manually designed scoring rule using observable search and engagement signals to prioritize pages.

The purpose of the baseline is to establish whether a simple, interpretable rule already provides useful prioritization before introducing a statistical model.

The Logistic Regression model is therefore evaluated against the baseline rather than being judged only by its absolute score.

### Validation design

The primary evaluation uses a client-grouped train/test split.

Grouping by `client_hash_id` ensures that no client appears in both the training and test sets.

This is a stricter evaluation than a random row split because pages from the same client can share characteristics. A random split could therefore produce an overly optimistic estimate if the model benefits from seeing other pages belonging to the same client during training.

The grouped test set is treated as the more credible estimate for the intended decision-support use.

### Leakage checks

I audited the five model features against the decision and outcome windows.

All five model features are calculated from the March decision window.

April impressions are used to construct the outcome label but are not included as a feature.

The label itself, future-period measurements, and derived future-change fields were excluded from the feature matrix.

A programmatic check confirmed that no future or label-derived fields were included among the model features.

This does not prove that every possible source of leakage is impossible, but it reduces the specific future-window leakage risk addressed in this analysis.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Chart 1 — Baseline vs model**

Precision@50

Baseline        █████████████████

Logistic Reg.   █████████████████████████

**Chart 2 — Feature coefficients**

| Feature                | Coefficient |
| ---------------------- | ----------: |
| `gsc_clicks`           |     -0.7651 |
| `sessions_organic`     |     +0.2404 |
| `gsc_impressions`      |     +0.1459 |
| `ga4_engaged_sessions` |     +0.0649 |
| `gsc_avg_position`     |     -0.0516 |


gsc_clicks had the largest absolute coefficient, indicating the strongest directional association among the standardized features in this model.

### Results

The Logistic Regression model achieved a Precision@50 of **0.66** on the client-grouped held-out test set, compared with **0.44** for the Week-4 baseline under the same evaluation design.

The model therefore provided stronger measured top-50 precision than the baseline in this evaluation.

The model also produced an Average Precision of **0.5903**, with overall precision of **0.6309** and recall of **0.0271** in the previously reported evaluation.

These metrics should be interpreted together. In particular, the low recall indicates that the model is not identifying most declining pages; its value is primarily in prioritizing a small set of pages for review.

The strongest standardized coefficient was associated with `gsc_clicks` (-0.7651), followed by `sessions_organic` (+0.2404). These coefficients describe associations learned by the model and should not be interpreted as causal effects.

## 5. Limitations

*What this work cannot claim.*

## 5. Limitations & Honest Framing

Several limitations constrain what can be concluded from this study.

First, the analysis uses a limited set of search and engagement signals. Other factors such as seasonality, search algorithm changes, competitor activity, search-intent changes, and content changes are not fully represented.

Second, the label describes an observed future decline in impressions. It does not identify why the decline occurred.

Third, the model estimates risk from historical relationships. The coefficients should not be interpreted as causal effects.

Fourth, the validation design groups by client, which reduces client-level leakage risk, but the analysis would be stronger with repeated time-based evaluations across multiple outcome windows.

Fifth, a Precision@50 result describes the top-ranked 50 pages in the evaluation set. It should not be interpreted as overall accuracy or as evidence that the model will perform identically in every future period.

Finally, the model is intended for decision support and human review rather than automated content changes.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

| Priority | Reason                 | Suggested action                       |
| -------- | ---------------------- | -------------------------------------- |
| HIGH     | LOW_SEARCH_CLICKS      | Review for potential refresh           |
| HIGH     | WEAKER_SEARCH_POSITION | Review search intent/content alignment |
| MEDIUM   | LOW_ENGAGEMENT         | Review before deciding                 |
| LOW      | COMBINED_SIGNALS       | Monitor                                |


## 6. Ranked Recommendations

The model output is converted into a ranked review queue.

### HIGH priority

Pages with higher estimated decline risk should be reviewed first.

The reviewer should investigate whether the page is outdated, mismatched with search intent, poorly aligned with the target query, or affected by another identifiable issue.

### MEDIUM priority

These pages warrant review but do not automatically justify a refresh.

### LOW priority

These pages can generally remain in monitoring unless other evidence suggests action is needed.

The ranking is intended to allocate human attention, not to automatically prescribe a content change.

**The "No-Go" box**


 What this model does NOT do:

*    Automatically publish changes
*    Automatically rewrite content

* Automatically delete pages
* Automatically redirect URLs
* Automatically declare content "bad"
* Establish causality
* Guarantee future performance


## 7. Reproducibility

The analysis is implemented in the project repository.

The workflow can be traced through the following notebooks:

- W01 — Research question
- W02 — ML task framing
- W03 — Data contract
- W03 — Feature leakage check
- W04 — Baseline score
- W05 — Model
- W06 — Validation audit
- W07 — Action playbook

The notebooks contain the SQL, Python code, validation logic, model configuration, and exported artifacts used in the analysis.

The repository is:

https://github.com/UmairMehfooz/Ml-Internship

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## 7. Artifacts the paper embeds

The final paper reuses the following artifacts generated during the analysis:

1. **Baseline vs. Logistic Regression**
   - Compares Precision@50 for the Week-4 baseline and Week-5 model.
   - Used in the Results section.

2. **Feature coefficient chart**
   - Shows the direction and relative magnitude of the standardized Logistic Regression coefficients.
   - Used to interpret the model signals.

3. **Validation comparison**
   - Compares the random row split with the client-grouped split.
   - Used to demonstrate why validation design matters.

4. **Ranked action queue**
   - Generated by the Week-7 action playbook.
   - Contains ranked pages, estimated decline risk, reason codes, and suggested actions.
   - Used in the Ranked Recommendations section.

5. **Metrics receipts**
   - JSON files containing the evaluation metrics and configuration used by the analysis.
   - Used to support reproducibility.

## Acknowledgments & Data Credit

This project was completed as part of the FlyRank ML Internship track.

**Built on the FlyRank ML Internship dataset.**

Data source: [FlyRank](https://flyrank.ai)

The analysis uses the publicly released internship dataset and does not reproduce private client names, URLs, search queries, or other private information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
